In [ ]:
!pip install catboost

In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

sns.set_theme(style="whitegrid")

In [ ]:
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(X.head())
print()

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер валидационной выборки: {X_val.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")

In [ ]:
MAX_TREES = 1000
LEARNING_RATE = 0.05

xgb_model = xgb.XGBRegressor(n_estimators=MAX_TREES, learning_rate=LEARNING_RATE, random_state=42)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_rmse = xgb_model.evals_result()['validation_0']['rmse']

lgb_model = lgb.LGBMRegressor(n_estimators=MAX_TREES, learning_rate=LEARNING_RATE, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='rmse')
lgb_metric_key = list(lgb_model.evals_result_['valid_0'].keys())[0]
lgb_rmse = lgb_model.evals_result_['valid_0'][lgb_metric_key]

cb_model = cb.CatBoostRegressor(iterations=MAX_TREES, learning_rate=LEARNING_RATE, eval_metric='RMSE', random_seed=42)
cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
cb_rmse = cb_model.evals_result_['learn']['RMSE']
cb_val_rmse = cb_model.evals_result_['validation']['RMSE']

plt.figure(figsize=(12, 6))
plt.plot(xgb_rmse, label='XGBoost Validation RMSE', color='red')
plt.plot(lgb_rmse, label='LightGBM Validation RMSE', color='blue')
plt.plot(cb_val_rmse, label='CatBoost Validation RMSE', color='green')

plt.title('Зависимость RMSE от количества деревьев', fontsize=14)
plt.xlabel('Количество деревьев', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
SELECTED_N_ESTIMATORS = 400

In [ ]:
class MyBoost:
  def __init__(self, n=400, lr=0.05, depth=7, subsample=1, colsample_bytree=1, smoothing=10, target_enc_coeff=1, seed=42) -> None:
      self.n = n
      self.lr = lr
      self.depth = depth
      self.seed = seed
      self.subsample = subsample
      self.colsample_bytree=colsample_bytree
      self.trees = []
      self.trees_cols = [] # храним фичи
      self.feature_names_ = []
      self.feature_importances_mat_ = []
      self.feature_importances_ = []
      self.cat_features_ = []
      self.cat_mappings = {} # средние для категорий
      
      # для целе-вароятностного кодирования
      self.smoothing = smoothing
      self.target_enc_coeff = target_enc_coeff

  def _target_category_weight(self, n: int):
    return 1/(1 + np.exp(-(n - self.target_enc_coeff)/self.smoothing))
  
  def _encode_categories(self, X, y=None, training=True):
      X_enc = X.copy()
      y_series = y
      
      if y is not None:
        y_series = pd.Series(y, index=X.index)
        
      cat_cols = X_enc.select_dtypes(include=['object', 'category', 'str']).columns
      
      for col in cat_cols:   
        if training:
          stats = y_series.groupby(X_enc[col]).agg(['count', 'mean'])
          smoothing_weight = self._target_category_weight(stats['count'])
          
          mapping = smoothing_weight * stats['mean'] + (1 - smoothing_weight) * self.initial_leaf
          self.cat_mappings[col] = mapping
          
        # вес * среднее_категории + (1 - вес) * среднее_глобальное
        # cat_mappings.get на случай если появятся новые категории  
        X_enc[col] = X_enc[col].map(self.cat_mappings.get(col, {})).fillna(self.initial_leaf)
      return X_enc
  
  def fit(self, X, y: np.ndarray):
    np.random.seed(self.seed)
    self.initial_leaf = y.mean()
    predictions = np.zeros(len(y)) + self.initial_leaf  
    
    X_encoded = self._encode_categories(X, y, training=True)
    
    self.feature_importances_mat_ = np.zeros(X.shape[1])
    self.feature_names_ = X.columns
    
    X_mat = X_encoded.values
    num_rows = len(X_mat)
    size = int(self.subsample * num_rows)
      
    num_cols = X_encoded.shape[1]
    cols_size = int(self.colsample_bytree * num_cols)
      
    for i in range(self.n):
      antigrad = y - predictions
       
      indices = np.random.choice(num_rows, size=size, replace=False)
      cols_indices = np.random.choice(num_cols, size=cols_size, replace=False)

      X_next = X_mat[indices][:, cols_indices]
      y_next = antigrad[indices]
      
      tree = DecisionTreeRegressor(max_depth=self.depth, random_state=self.seed + i, criterion="friedman_mse")

      tree.fit(X_next, y_next)
      self.trees.append(tree)
      self.trees_cols.append(cols_indices)

      self.feature_importances_mat_[cols_indices] += tree.feature_importances_
      
      predictions += tree.predict(X_mat[:, cols_indices]) * self.lr
      
    if self.n > 0:
        self.feature_importances_mat_ /= self.feature_importances_mat_.sum()
        self.feature_importances_ = pd.DataFrame(self.feature_importances_mat_, 
                                                 index=self.feature_names_,
                                                 columns=['importance']).sort_values(by='importance', ascending=False)
      

  def predict(self, samples):
    predictions = np.zeros(len(samples)) + self.initial_leaf
    
    samples_enc = self._encode_categories(samples, training=False)
    
    for i in range(self.n):
      cols = self.trees_cols[i]
      predictions += self.lr * self.trees[i].predict(samples_enc.values[:, cols])

    return predictions


In [ ]:
from sklearn.model_selection import ParameterGrid

X_train_full = pd.concat([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

grid_xgb = {
    'learning_rate': [0.05, 0.1],
    'max_depth':[5, 7],
    'subsample': [0.8, 1.0]
}

grid_lgb = {
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 127],
    'subsample':[0.8, 1.0]
}

grid_cat = {
    'learning_rate':[0.05, 0.1],
    'depth': [5, 7],
    'subsample':[0.8, 1.0]
}

def simple_grid_search(model_name, param_grid):
    print(f"--- Запуск Grid Search для {model_name} ---")
    best_rmse = float('inf')
    best_params = None

    start_time = time.time()

    for params in ParameterGrid(param_grid):
        if model_name == 'XGBoost':
            model = xgb.XGBRegressor(**params, n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1)
        elif model_name == 'LightGBM':
            model = lgb.LGBMRegressor(**params, n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1, verbose=-1)
        else:
            model = cb.CatBoostRegressor(**params, iterations=SELECTED_N_ESTIMATORS, random_seed=42, thread_count=-1, verbose=0)

        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = params

    total_time = time.time() - start_time
    print(f"Лучший Validation RMSE: {best_rmse:.4f}")
    print(f"Лучшие параметры: {best_params}\n")

    return best_params

best_params_dict = {}
best_params_dict['XGBoost'] = simple_grid_search('XGBoost', grid_xgb)
best_params_dict['LightGBM'] = simple_grid_search('LightGBM', grid_lgb)
best_params_dict['CatBoost'] = simple_grid_search('CatBoost', grid_cat)

In [ ]:
results = []

for name in['XGBoost', 'LightGBM', 'CatBoost', "MyBoost"]:
    params = best_params_dict.get(name)

    if name == 'XGBoost':
        model = xgb.XGBRegressor(n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1, **params)
    elif name == 'LightGBM':
        model = lgb.LGBMRegressor(n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1, verbose=-1, **params)
    elif name == 'MyBoost':
        model = MyBoost(subsample=0.9, colsample_bytree=0.7)
    else:
        model = cb.CatBoostRegressor(iterations=SELECTED_N_ESTIMATORS, random_seed=42, thread_count=-1, verbose=0, **params)

    start_time = time.time()
    model.fit(X_train_full, y_train_full)
    train_time = time.time() - start_time

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append({
        "Модель": name,
        "Время обучения": round(train_time, 3),
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2 Score": round(r2, 4)
    })
    
    if name == "MyBoost" :
        print(model.feature_importances_ )
        
df_results = pd.DataFrame(results)
display(df_results)

## Сверим модели

In [ ]:
import kagglehub
import pandas as pd
import seaborn as sb
import numpy as np

# Download latest version
path = kagglehub.dataset_download("kathuman/housing")

print("Path to dataset files:", path)
df = pd.read_csv(path + '/housing.csv')

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df.drop(['median_house_value'], axis=1),df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, stratify=df['ocean_proximity'], random_state=42)
y_train = y_train.values
y_test = y_test.values

### Произведем подбор параметров для нашей модели

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import root_mean_squared_error

param_grid_my = {
    'subsample': [0.7, 0.85, 1.0],        
    'colsample_bytree': [0.4, 0.7, 1.0],  
    'target_enc_coeff': [0.1],       
    'smoothing': [10],             
    'lr': [0.01],                    
    'depth': [10]                      
}

combinations = list(ParameterGrid(param_grid_my))

best_score = float('inf')
best_params = None

print(f"Всего комбинаций: {len(combinations)}")

for params in combinations:
    model = MyBoost(
        subsample=params['subsample'],
        target_enc_coeff=params['target_enc_coeff'],
        smoothing=params['smoothing'],
        lr=params['lr'],
        colsample_bytree=params['colsample_bytree'],
        depth=params['depth']
    )

    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    score = root_mean_squared_error(y_test, preds)
    
    print(f"Params: {params} | RMSE: {score:.4f}")
    
    if score < best_score:
        best_score = score
        best_params = params

In [ ]:
print(f"Best params: {best_params} | RMSE: {best_score:.4f}")

In [ ]:
best_model = MyBoost(
        subsample=best_params['subsample'],
        target_enc_coeff=best_params['target_enc_coeff'],
        smoothing=best_params['smoothing'],
        lr=best_params['lr'],
        colsample_bytree=best_params['colsample_bytree'],
        depth=best_params['depth']
    )
best_model.fit(X_train, y_train)
myboost_preds = best_model.predict(X_test)

best_model.feature_importances_

In [ ]:
catb = cb.CatBoostRegressor(cat_features=['ocean_proximity'])
catb.fit(X_train, y_train)
catb_preds = catb.predict(X_test)

catb_r = r2_score(y_test, catb_preds)
catb_rmse = root_mean_squared_error(y_test, catb_preds)
myboost_r = r2_score(y_test, myboost_preds)

print()
print(f'MyBoost r2 score: {myboost_r} | CatBoost r2 score: {catb_r}')
print(f'MyBoost rmse score: {best_score} | CatBoost rmse score: {catb_rmse}')


## Выводы
Я реализовал модель градиентного бустинга с фичами нативного кодирования признаков (target-encoding + сглаживание), случайная подвыборка признаков и валидационного датасета для каждого нового дерева, feature_importances_.

разница r2 примерно 0.03.
разница rmse примерно 450.
Но учится сильно дольше, тк операция подвыборки очень долгая (+- 30 секунд на обучение).